# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_02"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [7]:
# cele mai frecvente 15 canale sursă din dataset
# Notă: scoatem prefixul "@" pentru ca același canal scris cu/fără @ să fie numărat o singură dată.
df["source_channel"].str.lstrip("@").value_counts().head(15)

source_channel
RecorderRomania                  12177
CălinGeorgescu-CanalulOficial     6017
turcescu111                       5019
georgesimionoficial               3669
TuDecizi-s3g                       647
StareaNatiei                       623
AltcevacuAdrianArtene              363
roxindaniel                        305
otvdirect                          304
digi24hd56                         265
euronewsro                         238
DianaSosoacaOfficial               227
AdevaruriSecrete                   180
g4media479                         158
VeridicaRO                          91
Name: count, dtype: int64

In [8]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [9]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [10]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = """
Ești un analist specializat în analiza discursului politic românesc online, în special comentarii de pe rețele sociale (YouTube).

Sarcina ta este să analizezi un singur comentariu în limba română și să extragi structurat mai multe axe de analiză.

Reguli importante:
- Răspunzi DOAR cu JSON valid, fără text suplimentar înainte sau după.
- Folosești EXCLUSIV valorile din taxonomiile permise pentru câmpurile cu enum.
- Dacă nu poți determina o valoare în mod rezonabil, folosești "unclear" (sau null doar unde e specificat).
- Nu inventezi informație care nu apare în comentariu.
- Distingi clar între sentiment (cum sună comentariul în general) și stance (poziționarea față de țintă) — un comentariu poate fi negativ ca ton dar pro-țintă (ex: "ce nedreptate i se face!").
- Toate valorile text liber sunt în limba română.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic în limba română și identifică:

1. target: persoana, partidul sau entitatea principală despre care vorbește comentariul (text scurt în română, ex: "Călin Georgescu", "George Simion", "AUR", "sistemul"). Dacă nu există o țintă identificabilă, folosește null.

2. stance: poziționarea autorului față de target. Valori permise:
   - "pro" — susține / apără ținta
   - "contra" — critică / atacă ținta
   - "neutru" — menționează fără poziționare clară
   - "unclear" — nu se poate determina

3. sentiment: încărcătura emoțională generală a comentariului. Valori permise:
   - "pozitiv", "negativ", "mixt", "neutru"

4. tone: registrul dominant al exprimării. Valori permise:
   - "factual", "ironic", "agresiv", "emoțional", "religios", "conspirativ", "umoristic"

5. topic: tema principală a comentariului. Valori permise:
   - "alegeri", "justiție", "suveranitate", "religie", "economie", "media", "ordine_publică", "personal_atac", "personal_susținere", "altul"

6. interpretation_problem: principala dificultate de interpretare. Valori permise:
   - "none" — comentariul e clar
   - "sarcasm" — pare una, e alta
   - "ambiguitate" — sensul nu e clar
   - "ținte_multiple" — vorbește despre mai multe entități
   - "sentiment_vs_stance" — tonul și poziționarea diferă

7. rhetoric_type: figura retorică dominantă, dacă există. Valori permise:
   - "none", "apel_la_emoție", "apel_la_autoritate", "whataboutism", "dezumanizare", "idolatrizare", "victimizare"

Important:
- Returnează JSON valid cu EXACT aceste 7 chei: target, stance, sentiment, tone, topic, interpretation_problem, rhetoric_type.
- Nicio cheie extra, nicio cheie lipsă.
- Adaugă și un câmp "justification" cu o propoziție scurtă (max 20 cuvinte) care explică în română alegerile principale.

Format de răspuns așteptat:
{{
  "target": "...",
  "stance": "...",
  "sentiment": "...",
  "tone": "...",
  "topic": "...",
  "interpretation_problem": "...",
  "rhetoric_type": "...",
  "justification": "..."
}}

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [14]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [16]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [18]:
n_comments = 10
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,"```json\n{\n ""target"": ""Călin Georgescu"",\n ..."
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,"```json\n{\n ""target"": ""Călin Georgescu"",\n ..."
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,"```json\n{\n ""target"": ""autoritățile"",\n ""st..."
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,"```json\n{\n ""target"": ""cei care stau acolo s..."
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,"```json\n{\n ""target"": ""sistemul"",\n ""stance..."
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,"```json\n{\n ""target"": ""judecători"",\n ""stan..."
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","```json\n{\n ""target"": ""Noua Ordine Mondială""..."
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...","```json\n{\n ""target"": ""un hot corupt"",\n ""s..."
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...","```json\n{\n ""target"": ""Crin Antonescu"",\n ""..."
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,"```json\n{\n ""target"": ""firme din Galați"",\n ..."


# 9. Verificam rezultatele

In [19]:
results_df.model_output[0]

'```json\n{\n  "target": "Călin Georgescu",\n  "stance": "contra",\n  "sentiment": "negativ",\n  "tone": "ironic",\n  "topic": "personal_atac",\n  "interpretation_problem": "sentiment_vs_stance",\n  "rhetoric_type": "none",\n  "justification": "Comentariul critică ironic pe Călin Georgescu, sugerând că nu merită funcții înalte și nu ar trebui să fugă din țară."\n}\n```'

In [20]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [21]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,Călin Georgescu,contra,negativ,ironic,personal_atac,sentiment_vs_stance,
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,Călin Georgescu,pro,pozitiv,emoțional,personal_susținere,none,
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,autoritățile,contra,negativ,agresiv,ordine_publică,none,
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,cei care stau acolo sau trec pe acolo cu masina,pro,pozitiv,emoțional,altul,none,
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,sistemul,contra,negativ,emoțional,altul,sentiment_vs_stance,
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,judecători,contra,negativ,agresiv,justiție,none,
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...",Noua Ordine Mondială,contra,negativ,conspirativ,altul,none,
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...",un hot corupt,contra,negativ,agresiv,personal_atac,none,
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...",Crin Antonescu,contra,negativ,ironic,alegeri,none,
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,firme din Galați,neutru,neutru,factual,altul,none,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [ ]:
csv_output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.csv"
csv_output_file.parent.mkdir(parents=True, exist_ok=True)

parsed_df.to_csv(csv_output_file, index=False, encoding="utf-8-sig")
print("CSV salvat la:", csv_output_file)

CSV salvat la: /Users/catalinaminciuna/Library/CloudStorage/OneDrive-UniversitateaBabeş-Bolyai/masterat/inginerie AI/proiect AI Eng/echochamber-project-team3/outputs/student_02_prompt_outputs.csv


## 11. Reflecție asupra promptului

### Unde a funcționat bine promptul?
Promptul a identificat corect ținta principală pe 9 din 10 comentarii și a 
plasat corect tema (topic) în categoriile predefinite. La comentariile clar 
politice cu o singură țintă identificabilă — ex: rândul 8 ("Crin Alcoolescu") 
sau rândul 5 ("judecători") — clasificarea pe stance, sentiment și tone a fost 
plauzibilă și consistentă cu intuiția umană.

### Unde a eșuat?
La rândul 0 — "Multă sănătate dl. Președinte Călin Georgescu" — modelul a marcat 
`stance: contra`, deși comentariul e clar pro-CG. Probabil a interpretat formula 
de politețe ca ironie, în absența contextului cultural că în comunitatea pro-CG 
asemenea urări sunt sincere și frecvente.

### A confundat sentimentul cu stance-ul?
Pe 8 din 10 cazuri sentiment și stance au aceeași polaritate (`pro`+`pozitiv` 
sau `contra`+`negativ`). Asta sugerează că modelul tratează cele două axe ca 
sinonime, deși în prompt am specificat explicit că sunt diferite. La rândul 4 
modelul a marcat `interpretation_problem: "sentiment_vs_stance"`, dar tot a 
aliniat cele două coloane — semn că a sesizat tensiunea fără să știe cum să o codifice.

### Au creat probleme sarcasmul, ambiguitatea sau țintele multiple?
- **Sarcasmul invers** (politețe sinceră interpretată ca ironie) a apărut la 
  rândul 0 și a stricat clasificarea.
- **Ținte multiple** nu au apărut în acest eșantion — sample-ul aleator a prins 
  comentarii cu țintă unică.
- **Ambiguitate** — la rândul 3 ținta a devenit "cei care stau acolo sau trec pe 
  acolo cu mașina", o frază lungă în loc de o entitate concretă. Promptul ar 
  trebui să forțeze target-ul să fie o entitate scurtă sau `null`.

### Ce aș schimba la versiunea următoare a promptului?
1. **Few-shot examples** pentru distincția sentiment vs stance — un exemplu de 
   "ce nedreptate i se face!" cu `sentiment: negativ` + `stance: pro` ar ajuta 
   modelul să separe cele două axe.
2. **Constrângere pe lungimea target-ului** — maxim 5 cuvinte, altfel `null`.
3. **Eșantionare stratificată** pentru testare — sample-ul aleator a prins 
   comentarii ușor de clasificat. Pentru a evalua promptul corect, ar trebui 
   căutate explicit comentarii ironice, cu ținte multiple, sau cu polaritate 
   disociată.
4. **Testare pe un set mai mare** — 10 comentarii sunt prea puține pentru a 
   trage concluzii. Cu 100-200 s-ar putea calcula rate de confuzie sentiment 
   vs stance.